<a href="https://colab.research.google.com/github/DataFriend101/Machine_Learning/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [41]:
# Load dataset
import pandas as pd
df = pd.read_csv("content_refresh_anonymized.csv")

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

I prioritize content that already has search visibility but may have room for improvement. Pages with many impressions and weaker click engagement are ranked higher because improving pages that already receive attention could create a larger impact.

Reason codes:

- HIGH_VISIBILITY_REFRESH: The page receives strong search visibility and may be worth reviewing for improvement.
- LOW_CTR_OPPORTUNITY: The page receives impressions but has weaker click performance.
- REVIEW: The page does not strongly match the main opportunity signals but may still be worth checking.

In [35]:
# Signal 1: Volume
df["impression_bucket"] = pd.qcut(
    df["impressions_90d"],
    q=4,
    duplicates="drop"
)
volume_check = (
    df.groupby("impression_bucket", observed=True)
    .agg(
        n=("content_id", "count"),
        avg_clicks=("clicks_90d", "mean")
    )
    .reset_index()
)
volume_check

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


#### Volume signal verdict: Confirmed

Higher impression groups showed higher average clicks. This suggests that pages with stronger visibility may have a larger opportunity if improvements are made.




In [42]:
# Signal 2: CTR opportunity
df["ctr"] = (
    df["clicks_90d"] /
    df["impressions_90d"]
).fillna(0)

df["ctr_bucket"] = pd.qcut(
    df["ctr"],
    q=4,
    duplicates="drop"
)

ctr_check = (
    df.groupby("ctr_bucket", observed=True)
    .agg(
        n=("content_id", "count"),
        avg_impressions=("impressions_90d", "mean"),
        avg_clicks=("clicks_90d", "mean")
    )
    .reset_index()
)

ctr_check

,ctr_bucket,n,avg_impressions,avg_clicks
0,"(-0.001, 0.000697]",15000,1790.549333,0.588667
1,"(0.000697, 0.00285]",7500,9359.844000,15.637467
2,"(0.00285, 1.0]",7500,7860.522533,47.574533


#### CTR signal verdict: Mixed

Some pages have strong visibility but weaker CTR, which may indicate optimization opportunities. However, CTR alone does not explain whether the content itself needs improvement

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

To create the ranked queue, I combine the signals from my rule into a simple score. The score is designed to prioritize pages that already have visibility but may have room for improvement.

A higher score means the page may be a stronger candidate for review. The score is only used for prioritization and is not a prediction of future performance.

In [43]:
# Normalize visibility so larger pages receive higher scores
df["visibility_score"] = (
    df["impressions_90d"] /
    df["impressions_90d"].max()
)
# Pages with lower CTR relative to the dataset receive a higher opportunity score
df["ctr_opportunity"] = (
    1 - (df["ctr"] / df["ctr"].max())
)
df[[
    "impressions_90d",
    "ctr",
    "visibility_score",
    "ctr_opportunity"
]].head()

,impressions_90d,ctr,visibility_score,ctr_opportunity
0,3803,0.007626,0.007346,0.992374
1,15320,0.000457,0.029592,0.999543
2,12581,0.000874,0.024301,0.999126
3,11751,0.004936,0.022698,0.995064
4,19140,0.001254,0.036970,0.998746


In [44]:
# Combine signals into one transparent score
df["score"] = (
    df["visibility_score"] *
    df["ctr_opportunity"]
)
df["score"].describe()

,score
count,30000.000000
mean,0.010014
std,0.032423
min,0.000000
25%,0.000156
50%,0.001410
75%,0.006966
max,0.998569


In [45]:
# Define simple thresholds
high_visibility_threshold = df["impressions_90d"].quantile(0.75)
low_ctr_threshold = df["ctr"].median()

def assign_reason(row):
    if (
        row["impressions_90d"] >= high_visibility_threshold
        and row["ctr"] <= low_ctr_threshold
    ):
        return "LOW_CTR_OPPORTUNITY"
    elif row["impressions_90d"] >= high_visibility_threshold:
        return "HIGH_VISIBILITY_REFRESH"
    else:
        return "REVIEW"
df["reason_code"] = df.apply(assign_reason, axis=1)
df["reason_code"].value_counts()

,count
reason_code,
REVIEW,22500
HIGH_VISIBILITY_REFRESH,6295
LOW_CTR_OPPORTUNITY,1205


In [46]:
# Add the action labels
def assign_action(reason):
    if reason == "LOW_CTR_OPPORTUNITY":
        return "OPTIMIZE_SNIPPET"
    elif reason == "HIGH_VISIBILITY_REFRESH":
        return "REVIEW_UPDATE"
    else:
        return "MONITOR"

df["action"] = df["reason_code"].apply(assign_action)
df["action"].value_counts()

,count
action,
MONITOR,22500
REVIEW_UPDATE,6295
OPTIMIZE_SNIPPET,1205


In [47]:
# create ranked queue
baseline_queue = (
    df[
        [
            "content_id",
            "score",
            "reason_code",
            "action"
        ]
    ]
    .sort_values(
        "score",
        ascending=False
    )
)
baseline_queue.head(10)

,content_id,score,reason_code,action
6653,content_5fe46e04994d,0.998569,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE
17812,content_aaef01a50def,0.996376,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE
26844,content_8c19996aa890,0.982137,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE
19636,content_2cb567c3c89b,0.960451,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE
21819,content_4c36c775b818,0.890865,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE
29400,content_2dba2b1f9536,0.854764,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE
29879,content_1a9e894be2e2,0.802055,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE
13537,content_2c2606c5d176,0.667443,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE
18870,content_db5989a78dd3,0.665188,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE
14090,content_44e481c8f55b,0.600064,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE


In [48]:
# Create the CSV
import os
os.makedirs(
    "work/outputs",
    exist_ok=True
)
baseline_queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [53]:
baseline_queue.head(20)

,content_id,score,reason_code,action
6653,content_5fe46e04994d,0.998569,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE
17812,content_aaef01a50def,0.996376,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE
26844,content_8c19996aa890,0.982137,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE
19636,content_2cb567c3c89b,0.960451,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE
21819,content_4c36c775b818,0.890865,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE
29400,content_2dba2b1f9536,0.854764,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE
29879,content_1a9e894be2e2,0.802055,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE
13537,content_2c2606c5d176,0.667443,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE
18870,content_db5989a78dd3,0.665188,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE
14090,content_44e481c8f55b,0.600064,HIGH_VISIBILITY_REFRESH,REVIEW_UPDATE



- HIGH_VISIBILITY_REFRESH:
  - Action: REVIEW_UPDATE
  - Why: These pages receive many impressions, so improvements could have a larger potential impact
  - Confidence: Medium, because visibility suggests opportunity but does not guarantee a content issue
  - What would make it wrong: The page may already satisfy users or the traffic may come from queries where updates would not improve performance

- LOW_CTR_OPPORTUNITY:
  - Action: OPTIMIZE_SNIPPET
  - Why: These pages receive impressions but have weaker click performance.
  - Confidence: Medium, because CTR is only one engagement signal.
  - What would make it wrong: Lower CTR may be caused by search intent differences rather than the page itself.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some of the weaker picks are pages selected mainly because they have high visibility. High impressions indicate potential impact, but they do not guarantee that the page actually needs improvement. A page could already be performing well, or the reason for lower engagement could come from search intent rather than the content itself.

### Leakage check
I confirmed that the score only uses historical performance features:
- impressions_90d
- clicks_90d
- calculated CTR

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.